In [1]:
import pandas as pd
import os
os.chdir("..")
from app.load_claims import load_claims

In [2]:
df = load_claims()
display(df.head())
print(df.shape)
print(df.dtypes)

/Users/electricalman/Desktop/internship_prep/gcp-intership-prep/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,claim_id,provider_id,payer_id,patient_age_bucket,service_line,procedure_group,claim_amount,allowed_amount,patient_responsibility,days_to_submit,prior_auth_required,prior_auth_present,eligibility_verified,submitted_at,claim_status,was_denied
0,C180,PROV_1,PAYER_1,senior,surgery,complex,6760.0,5660.0,250.0,0,1,1,1,2025-11-26,denied,1
1,C630,PROV_11,PAYER_1,senior,imaging,diagnostic,5410.0,5610.0,500.0,0,1,1,1,2025-09-02,denied,1
2,C900,PROV_1,PAYER_1,senior,surgery,complex,6400.0,6980.0,510.0,0,1,1,1,2025-12-06,denied,1
3,C360,PROV_1,PAYER_1,senior,surgery,complex,4420.0,4240.0,490.0,0,1,1,1,2025-05-30,denied,1
4,C450,PROV_11,PAYER_1,senior,imaging,diagnostic,7750.0,7030.0,260.0,0,1,1,1,2026-03-01,denied,1


(10000, 16)
claim_id                   object
provider_id                object
payer_id                   object
patient_age_bucket         object
service_line               object
procedure_group            object
claim_amount              float64
allowed_amount            float64
patient_responsibility    float64
days_to_submit              Int64
prior_auth_required         Int64
prior_auth_present          Int64
eligibility_verified        Int64
submitted_at               dbdate
claim_status               object
was_denied                  Int64
dtype: object


In [3]:
print(f"The denial rate is: {df['was_denied'].mean()}")

The denial rate is: 0.2993


This helps me find the payers that have highe denial rate

In [4]:
df.groupby("payer_id")["was_denied"].mean()

payer_id
PAYER_1    0.496496
PAYER_2    0.498758
PAYER_3    0.500502
PAYER_4         0.0
PAYER_5         0.0
Name: was_denied, dtype: Float64

In [5]:
df.groupby("prior_auth_required")["was_denied"].mean()

prior_auth_required
0    0.200519
1    0.398358
Name: was_denied, dtype: Float64

Service line denial rates

In [6]:
df.groupby("service_line")["was_denied"].mean()

service_line
emergency       0.200792
imaging         0.395875
primary_care    0.200242
surgery          0.40089
Name: was_denied, dtype: Float64

In [7]:
df.groupby("claim_status")["was_denied"].count()

claim_status
denied    2993
paid      7007
Name: was_denied, dtype: Int64

This checks helps me verify columns with potential data leakage

In [8]:
df.groupby("claim_status")["was_denied"].count()

claim_status
denied    2993
paid      7007
Name: was_denied, dtype: Int64

The target is not severely imbalanced. About 58% of claims were not denied and 42% were denied, so the model will have a reasonable number of examples from both classes. Accuracy is still not enough by itself, but imbalance is not a major issue in this synthetic dataset.

If it were 95% not denied and 5% denied, imbalance would be a huge problem.
At 58/42, the target is close enough to balanced for a baseline model.

In [9]:
df["was_denied"].value_counts(normalize=True)

was_denied
0    0.7007
1    0.2993
Name: proportion, dtype: Float64

In [10]:
df["claim_amount"].min()

np.float64(100.0)

In [11]:
df["claim_amount"].max()


np.float64(9099.0)

In [12]:
(df["claim_amount"] > 100).sum()

np.int64(9999)

1. Are high claim amounts denied more often?

In [13]:
df["claim_bucket_bucket"] = pd.cut(
    df["claim_amount"],
    bins=[0,500,5000,7500, float("inf")],
    labels=["low","medium", "high", "very_high"]
)

df.groupby("claim_bucket_bucket")["was_denied"].mean()

/var/folders/5s/wp7dyzq57dvbmqdyw5535vtm0000gn/T/ipykernel_52279/2163954702.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("claim_bucket_bucket")["was_denied"].mean()


claim_bucket_bucket
low          0.299363
medium       0.298516
high         0.300288
very_high    0.299943
Name: was_denied, dtype: Float64

2. Does missing prior authorization increase denial rate?

In [14]:
df.groupby(["prior_auth_required", "prior_auth_present"])["was_denied"].agg(["count", "mean"]).reset_index()

,prior_auth_required,prior_auth_present,count,mean
0,0,0,1681,0.196907
1,0,1,3326,0.202345
2,1,0,1664,0.40024
3,1,1,3329,0.397417


3. Does lack of eligibility verification increase denial rate?

In [15]:
df.groupby("eligibility_verified")["was_denied"].agg(["count", "mean"]).reset_index()

,eligibility_verified,count,mean
0,0,2005,0.0
1,1,7995,0.374359


In [16]:
df.groupby(["claim_status", "was_denied"]).size().reset_index(name="count")

,claim_status,was_denied,count
0,denied,1,2993
1,paid,0,7007
